In [1]:
!pip install optuna mlflow dagshub seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.4/263.4 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import optuna
import json
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

import dagshub
import mlflow
import mlflow.sklearn

In [3]:
dagshub.init(
    repo_owner="Aryanupadhyay23",
    repo_name="Twitter-Sentiment-Analysis-MLOps",
    mlflow=True
)

mlflow.set_tracking_uri(
    "https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow"
)

mlflow.set_experiment("hyperparameter tuning")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=c8dc4571-e2bf-4d6a-91c7-bba3f85d5fca&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=611689b1f1271df178147b1af7a5da6f66853d075419b619d2ce913e3ed1247e




Accessing as Aryanupadhyay23

Initialized MLflow to track repo "Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps"

Repository Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps initialized!

<Experiment: artifact_location='mlflow-artifacts:/e8aa851b064b48618f790743afea7af8', creation_time=1771822080177, experiment_id='6', last_update_time=1771822080177, lifecycle_stage='active', name='hyperparameter tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [4]:
df = pd.read_csv("twitter_cleaned.csv")

df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])

print("Classes:", label_encoder.classes_)

Classes: ['negative' 'neutral' 'positive']


In [5]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment_encoded"],
    test_size=0.2,
    stratify=df["sentiment_encoded"],
    random_state=42
)

y_train = np.array(y_train)
y_test = np.array(y_test)

In [6]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    min_df=2
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (41292, 8000)
Test shape: (10323, 8000)


In [11]:
def build_sgd(trial):

    loss = trial.suggest_categorical(
        "loss", ["hinge", "log_loss", "modified_huber"]
    )

    alpha = trial.suggest_float("alpha", 1e-6, 1e-2, log=True)

    penalty = trial.suggest_categorical(
        "penalty", ["l2", "l1", "elasticnet"]
    )

    return SGDClassifier(
        loss=loss,
        alpha=alpha,
        penalty=penalty,
        learning_rate="optimal",  # fixed
        class_weight="balanced",
        max_iter=2000,
        tol=1e-3,
        random_state=42,
        n_jobs=-1
    )

In [12]:
def objective(trial):

    model = build_sgd(trial)

    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        macro = f1_score(y_test, preds, average="macro")
        weighted = f1_score(y_test, preds, average="weighted")
        acc = accuracy_score(y_test, preds)

        mlflow.log_params(trial.params)
        mlflow.log_metric("macro_f1", macro)
        mlflow.log_metric("weighted_f1", weighted)
        mlflow.log_metric("accuracy", acc)

    return macro

In [13]:
study = optuna.create_study(
    direction="maximize",
    study_name="sgd_study",
    storage="sqlite:///sgd_optuna.db",
    load_if_exists=True
)

[I 2026-02-23 09:25:43,036] Using an existing study with name 'sgd_study' instead of creating a new one.


In [14]:
with mlflow.start_run(run_name="SGDClassifier"):

    mlflow.log_param("model_name", "SGDClassifier")
    mlflow.log_param("vectorizer", "CountVectorizer")
    mlflow.log_param("ngram_range", "(1,2)")
    mlflow.log_param("max_features", 8000)
    mlflow.log_param("min_df", 2)
    mlflow.log_param("train_samples", X_train.shape[0])
    mlflow.log_param("num_features", X_train.shape[1])

    # Hyperparameter tuning
    study.optimize(objective, n_trials=50)

    best_params = study.best_params

    print("Best Macro F1:", study.best_value)
    print("Best Params:", best_params)

    mlflow.log_params(best_params)
    mlflow.log_metric("best_single_split_macro_f1", study.best_value)

    # Train final model
    final_model = SGDClassifier(
        **best_params,
        class_weight="balanced",
        max_iter=2000,
        tol=1e-3,
        random_state=42,
        n_jobs=-1
    )

    final_model.fit(X_train, y_train)

    # 5-Fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    cv_macro = []
    cv_acc = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model = SGDClassifier(
            **best_params,
            class_weight="balanced",
            max_iter=2000,
            tol=1e-3,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        macro = f1_score(y_val, preds, average="macro")
        acc = accuracy_score(y_val, preds)

        cv_macro.append(macro)
        cv_acc.append(acc)

        print(f"Fold {fold} - Macro F1: {macro:.4f}")

    mlflow.log_metric("cv_macro_f1_mean", np.mean(cv_macro))
    mlflow.log_metric("cv_macro_f1_std", np.std(cv_macro))
    mlflow.log_metric("cv_accuracy_mean", np.mean(cv_acc))

    # Final test evaluation
    preds_test = final_model.predict(X_test)

    test_macro = f1_score(y_test, preds_test, average="macro")
    test_weighted = f1_score(y_test, preds_test, average="weighted")
    test_accuracy = accuracy_score(y_test, preds_test)

    mlflow.log_metric("test_macro_f1", test_macro)
    mlflow.log_metric("test_weighted_f1", test_weighted)
    mlflow.log_metric("test_accuracy", test_accuracy)

    print("Final Test Macro F1:", test_macro)

    # Artifacts
    report = classification_report(y_test, preds_test, output_dict=True)
    with open("classification_report_sgd.json", "w") as f:
        json.dump(report, f, indent=4)
    mlflow.log_artifact("classification_report_sgd.json")

    cm = confusion_matrix(y_test, preds_test)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix - SGDClassifier")
    plt.savefig("confusion_matrix_sgd.png")
    plt.close()
    mlflow.log_artifact("confusion_matrix_sgd.png")

    study.trials_dataframe().to_csv("optuna_trials_sgd.csv", index=False)
    mlflow.log_artifact("optuna_trials_sgd.csv")

    mlflow.sklearn.log_model(final_model, artifact_path="model")

print("SGDClassifier experiment completed successfully.")

🏃 View run trial_2 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/e7e8531d04664b7291049859c2a8cb01
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 09:25:48,795] Trial 2 finished with value: 0.7798990140123684 and parameters: {'loss': 'log_loss', 'alpha': 5.674807868518566e-05, 'penalty': 'elasticnet'}. Best is trial 0 with value: 0.8029359835409279.
[I 2026-02-23 09:26:06,181] Trial 3 finished with value: 0.7896087382154753 and parameters: {'loss': 'modified_huber', 'alpha': 2.2366235102958972e-06, 'penalty': 'l1'}. Best is trial 0 with value: 0.8029359835409279.


🏃 View run trial_3 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/b787f1e7081644c286c53c6281438926
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 09:26:10,033] Trial 4 finished with value: 0.8034805134072368 and parameters: {'loss': 'modified_huber', 'alpha': 1.0011482197406864e-05, 'penalty': 'elasticnet'}. Best is trial 4 with value: 0.8034805134072368.


🏃 View run trial_4 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/3c543de87005479bbc421ea2ff005973
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6
🏃 View run trial_5 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/2197bea1a13342c8b5579a0cb300f29a
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


...

🏃 View run trial_51 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/46d7d5af1b6f4e5fa6709e4ddfc42372
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 09:31:40,550] Trial 51 finished with value: 0.8043166444595257 and parameters: {'loss': 'log_loss', 'alpha': 1.01332894077273e-05, 'penalty': 'l2'}. Best is trial 36 with value: 0.8098629606233742.


Best Macro F1: 0.8098629606233742
Best Params: {'loss': 'hinge', 'alpha': 6.089647386172834e-06, 'penalty': 'l2'}
Fold 1 - Macro F1: 0.7969
Fold 2 - Macro F1: 0.7945
Fold 3 - Macro F1: 0.7954
Fold 4 - Macro F1: 0.7970
Fold 5 - Macro F1: 0.7897
Final Test Macro F1: 0.8098629606233742


2026/02/23 09:31:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 09:32:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run SGDClassifier at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/fc2cc0b6b933458db3ebd5f0470b634d
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6
SGDClassifier experiment completed successfully.
